In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score
from scipy import stats
import pickle

# 1. Загружаем тестовые данные
df = pd.read_csv("../data/UCI_Credit_Card.csv")
X = df.iloc[:, 1:-1]
y_true = df.iloc[:, -1]

from sklearn.model_selection import train_test_split
_, X_test, _, y_test = train_test_split(
    X, y_true, test_size=0.2, random_state=42, stratify=y_true
)

# 2. Загружаем обе модели
with open("../models/rf_default_model.pkl", "rb") as f:
    model_v1 = pickle.load(f)

with open("../models/rf_default_model_v2.pkl", "rb") as f:
    model_v2 = pickle.load(f)

# 3. Предсказания
y_pred_v1 = model_v1.predict(X_test)
y_pred_v2 = model_v2.predict(X_test)

# 4. Количество положительных предсказаний
n1 = np.sum(y_pred_v1 == 1)
n2 = np.sum(y_pred_v2 == 1)

# 5. Precision
p1 = precision_score(y_test, y_pred_v1)
p2 = precision_score(y_test, y_pred_v2)

# 6. z-test для двух пропорций
p = (p1 * n1 + p2 * n2) / (n1 + n2)
se = np.sqrt(p * (1 - p) * (1/n1 + 1/n2))
z = (p1 - p2) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print("Результаты z-test для A/B-теста")
print(f"Тестовая выборка: {len(y_test)} объектов")
print(f"Положительных предсказаний v1: {n1}")
print(f"Положительных предсказаний v2: {n2}")
print(f"Precision v1: {p1:.4f}")
print(f"Precision v2: {p2:.4f}")
print(f"z-статистика: {z:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("Различия статистически значимы (p < 0.05)")
else:
    print("Различия статистически не значимы (p >= 0.05)")

Результаты z-test для A/B-теста
Тестовая выборка: 6000 объектов
Положительных предсказаний v1: 717
Положительных предсказаний v2: 722
Precision v1: 0.7950
Precision v2: 0.6579
z-статистика: 5.8310
p-value: 0.000000
Различия статистически значимы (p < 0.05)
